In [ ]:
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(42)

THETA = 0.4
EPS = 0.1
ETA = 0.2
SAMPLE_SIZES = [5, 10, 20, 50, 100, 200]
N_REPS = 1000
N_TEST = 200_000

In [ ]:
def target(x, theta=THETA):
    return (x > theta).astype(int)

def generate_data(m, eta=0.0, theta=THETA, rng=RNG):
    x = rng.uniform(0.0, 1.0, m)
    y = target(x, theta)
    if eta > 0:
        flip = rng.random(m) < eta
        y = np.where(flip, 1 - y, y)
    return x, y

demo_rng = np.random.default_rng(0)
x_demo, y_demo = generate_data(50, rng=demo_rng)
x_demo_n, y_demo_n = generate_data(50, eta=ETA, rng=demo_rng)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for ax, (xs, ys, title) in zip(axes, [
    (x_demo, y_demo, "Realizable (m=50)"),
    (x_demo_n, y_demo_n, f"Agnostic, eta={ETA} (m=50)"),
]):
    ax.scatter(xs, ys, c=ys, cmap="coolwarm", edgecolors="k", s=40)
    ax.axvline(THETA, color="green", linestyle="--", label=f"theta={THETA}")
    ax.set_xlabel("x")
    ax.set_title(title)
    ax.legend()
axes[0].set_ylabel("label y")
plt.tight_layout()
plt.show()
# Realizable: labels switch from 0 to 1 exactly at theta, so a threshold separates the data perfectly.
# Agnostic: about 20% of labels are flipped, so some points sit on the wrong side of theta and no threshold fits all of them.

In [ ]:
def predict(x, t):
    return (x > t).astype(int)

def erm_threshold(x, y):
    # Thresholds inside the same gap between consecutive sorted points give the same
    # training error, so testing one candidate per gap (its midpoint) plus t = 0 and t = 1
    # covers every possible training error.
    xs = np.sort(x)
    ys = y[np.argsort(x)]
    candidates = np.concatenate(([0.0], (xs[:-1] + xs[1:]) / 2.0, [1.0]))
    preds = (xs[None, :] > candidates[:, None]).astype(int)
    n_errors = (preds != ys[None, :]).sum(axis=1)
    return candidates[np.argmin(n_errors)]

check_rng = np.random.default_rng(1)
x_s, y_s = generate_data(30, rng=check_rng)
t_hat = erm_threshold(x_s, y_s)
print(f"Learned threshold: {t_hat:.4f} (true theta = {THETA})")
print(f"Training error: {np.mean(predict(x_s, t_hat) != y_s):.3f}")

In [ ]:
def make_test_set(eta, n=N_TEST, seed=12345):
    rng = np.random.default_rng(seed)
    x = rng.uniform(0.0, 1.0, n)
    y = target(x)
    if eta > 0:
        flip = rng.random(n) < eta
        y = np.where(flip, 1 - y, y)
    return x, y

def run_trial(m, eta, x_test, y_test):
    x, y = generate_data(m, eta)
    t = erm_threshold(x, y)
    emp = np.mean(predict(x, t) != y)
    true = np.mean(predict(x_test, t) != y_test)
    return t, emp, true

x_test_r, y_test_r = make_test_set(0.0)
print("m | learned t | empirical err | true err | |t - theta|")
for m in SAMPLE_SIZES:
    t, emp, true = run_trial(m, 0.0, x_test_r, y_test_r)
    print(f"{m:4d} | {t:.4f} | {emp:.3f} | {true:.4f} | {abs(t - THETA):.4f}")
# Empirical error is 0 for every m: the class contains the target, so ERM always fits the training set perfectly.
# True error is close to |t - theta|, the exact error of a threshold under the uniform distribution, and shrinks as m grows.

In [ ]:
def run_setting(eta, n_reps=N_REPS):
    x_test, y_test = make_test_set(eta)
    return {m: np.array([run_trial(m, eta, x_test, y_test) for _ in range(n_reps)])
            for m in SAMPLE_SIZES}

def summarize(results, eta):
    rows = []
    for m in SAMPLE_SIZES:
        r = results[m]
        emp, true = r[:, 1], r[:, 2]
        rows.append({
            "m": m,
            "emp_mean": emp.mean(),
            "true_mean": true.mean(),
            "P(emp=0)": np.mean(emp == 0),
            "P(L-bayes>eps)": np.mean(true - eta > EPS),
        })
    return rows

res_real = run_setting(0.0)
rows_real = summarize(res_real, 0.0)
for row in rows_real:
    print({k: round(float(v), 4) for k, v in row.items()})
# Realizable: the mean true error falls from about 0.10 at m=5 to about 0.002 at m=200, roughly halving each time m doubles.
# Pr(L(h) > eps) falls from 0.31 at m=5 to 0.10 at m=10, 0.007 at m=20, and 0 from m=50 on.

In [ ]:
ms = np.array(SAMPLE_SIZES)
p_real = [r["P(L-bayes>eps)"] for r in rows_real]

plt.figure(figsize=(6, 4))
plt.plot(ms, p_real, marker="o", label="Realizable")
plt.xscale("log")
plt.xlabel("m (log scale)")
plt.ylabel(f"Pr(L(h) > {EPS})")
plt.title("PAC: probability of failure vs sample size")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()
# The failure probability goes to 0 as m grows: with enough examples, ERM returns an eps-accurate hypothesis with high probability, which is the PAC guarantee.

In [ ]:
res_agn = run_setting(ETA)
rows_agn = summarize(res_agn, ETA)
for row in rows_agn:
    print({k: round(float(v), 4) for k, v in row.items()})
# Agnostic: the mean empirical error rises from about 0.12 at m=5 to about 0.20 at m=200, approaching eta from below.
# The empirical error never reaches zero for large m: P(emp=0) is 0.49 at m=5, 0.18 at m=10, 0.02 at m=20, and 0 from m=50 on.
# The mean true error falls from about 0.32 to about 0.21, approaching eta from above. It cannot go below eta, because no threshold in H does better than eta.
# Pr(L(h) - eta > eps) falls from 0.45 at m=5 to 0.29 at m=10, 0.17 at m=20, 0.04 at m=50, and 0.005 at m=100.

In [ ]:
p_agn = [r["P(L-bayes>eps)"] for r in rows_agn]
emp_real = [r["emp_mean"] for r in rows_real]
true_real = [r["true_mean"] for r in rows_real]
emp_agn = [r["emp_mean"] for r in rows_agn]
true_agn = [r["true_mean"] for r in rows_agn]
zero_real = [r["P(emp=0)"] for r in rows_real]
zero_agn = [r["P(emp=0)"] for r in rows_agn]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(ms, emp_real, "o-", label="empirical (realizable)")
axes[0].plot(ms, true_real, "s-", label="true (realizable)")
axes[0].plot(ms, emp_agn, "o--", label="empirical (agnostic)")
axes[0].plot(ms, true_agn, "s--", label="true (agnostic)")
axes[0].axhline(ETA, color="gray", linestyle=":", label=f"eta={ETA}")
axes[0].set_xscale("log")
axes[0].set_xlabel("m")
axes[0].set_ylabel("error")
axes[0].set_title("Mean errors")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].plot(ms, p_real, "o-", label="realizable")
axes[1].plot(ms, p_agn, "s--", label="agnostic (excess risk)")
axes[1].set_xscale("log")
axes[1].set_xlabel("m")
axes[1].set_ylabel(f"Pr(excess > {EPS})")
axes[1].set_title("Probability of failure")
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].plot(ms, zero_real, "o-", label="realizable")
axes[2].plot(ms, zero_agn, "s--", label="agnostic")
axes[2].set_xscale("log")
axes[2].set_xlabel("m")
axes[2].set_ylabel("P(empirical error = 0)")
axes[2].set_title("Can empirical error reach zero?")
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()
# Left: in both settings the empirical and true errors move toward each other. With noise they converge to eta, not to 0.
# Middle: the failure probability goes to 0 in both settings, but the agnostic one decreases more slowly at small m, because the sample size needed scales with 1/eps^2 instead of 1/eps.
# Right: without noise, zero training error is reached for every m. With noise, zero training error becomes impossible once m is large enough.

In [ ]:
# Conclusion
# Realizable (PAC): the target is in H, so ERM reaches zero training error and the true error goes to 0 as m grows. Pr(L(h) > eps) reaches 0 from m = 50.
# Agnostic (APAC): label noise makes zero training error impossible for large m. The true error stays above eta = 0.2, because no hypothesis in H beats eta.
# What converges in the agnostic setting is the excess risk L(h) - eta, and it satisfies the agnostic PAC guarantee L(h) <= min L(h') + eps.
# Agnostic PAC is harder: more data is needed for the same eps (about 1/eps^2 instead of 1/eps), visible in the slower decrease of Pr(excess > eps).